# 60 Visualize MediaPipe detections on S8

Render Subject 12 (S8, bare-cap) from the same 21-view sweep used in notebooks 56 and 59, run MediaPipe Face Detector on every view, and draw the detected bounding box (with confidence score) on top of each rendered view. This makes it possible to see exactly what BlazeFace is locking onto when it fires on the bare-cap cohort.

Three variants are rendered for visual comparison:

- **Original**: head-isolated CTF mesh straight out of `run_pipeline` (no anonymization).
- **Deletion**: same mesh after `delete_masked_vertices` (the shipped operator).
- **Noise**: same mesh after the rejected noise-perturbation strawman (`noise_perturb_with_taper`).

All three share head-isolation and CTF alignment from the same in-memory pipeline run, so the only factor that varies across variants is the operator applied. Detection bounding boxes are drawn in green; views with no detection are left clean.

Output: `thesis_results_out/s8_mediapipe_boxes/{variant}_grid.png` (3 contact sheets, one per variant).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_data import s_id, load_raw, load_landmarks
from _thesis_pipeline import run_pipeline
from _validator_noise import noise_perturb_with_taper
from _validator_render import (
    YAWS, PITCHES,
    render_views, mediapipe_face_detector, mediapipe_detect_with_boxes,
)

import pandas as pd
import cv2
import matplotlib.pyplot as plt

import cedalion.dataclasses as cdc

SUBJECT = 12  # S8
OUT_DIR = pathlib.Path('thesis_results_out') / 's8_mediapipe_boxes'
OUT_DIR.mkdir(parents=True, exist_ok=True)

_fd = mediapipe_face_detector()

## 1. Helpers

Render geometry and MediaPipe detection are shared with notebook 72
via `_validator_render`: `render_views(surface, out_dir, tag)`
produces the 21-view yaw/pitch sweep at the canonical 700 mm camera
distance, and `mediapipe_detect_with_boxes(image_path, detector)`
returns the annotated BGR image plus per-detection
`(xmin, ymin, w, h, score)` boxes.

## 2. Run all three variants on S8

In [ ]:
raw = load_raw(SUBJECT)
lm = load_landmarks(SUBJECT)
art = run_pipeline(raw, lm, subject=SUBJECT)
lm_ctf_xyz = art.landmarks_ctf.pint.dequantify().values

noise_mesh = noise_perturb_with_taper(art.surface_ctf.mesh, art.mask, lm_ctf_xyz)
surface_noise = cdc.TrimeshSurface(
    mesh=noise_mesh,
    crs=art.surface_ctf.crs,
    units=art.surface_ctf.units,
)

variants = {
    'original': art.surface_ctf,
    'delete':   art.surface_anon_ctf,
    'noise':    surface_noise,
}

all_rows = []
annotated_paths = {tag: {} for tag in variants}

for tag, surface in variants.items():
    print(f'\n=== {tag} ===', flush=True)
    sub_dir = OUT_DIR / tag
    files = render_views(surface, sub_dir, tag)
    annot_dir = sub_dir / 'annotated'
    annot_dir.mkdir(exist_ok=True)
    for yaw, pitch, fn in files:
        annotated, boxes = mediapipe_detect_with_boxes(fn, _fd)
        out_fn = annot_dir / fn.name
        cv2.imwrite(str(out_fn), annotated)
        annotated_paths[tag][(yaw, pitch)] = out_fn
        if boxes:
            for (x, y, w, h, score) in boxes:
                all_rows.append({
                    's_id': s_id(SUBJECT), 'subject': SUBJECT,
                    'variant': tag, 'yaw': yaw, 'pitch': pitch,
                    'bbox_x': x, 'bbox_y': y, 'bbox_w': w, 'bbox_h': h,
                    'score': score,
                })
        print(f'  yaw={yaw:+4d} pitch={pitch:+3d}  hits={len(boxes)}'
              f'  scores={[round(b[4], 2) for b in boxes]}')

df = pd.DataFrame(all_rows)
csv_out = OUT_DIR / 's8_detections.csv'
df.to_csv(csv_out, index=False)
print(f'\nWrote {csv_out} with {len(df)} detections across the three variants.')

## 4. Build a 3 x 21 contact sheet

Rows: original / delete / noise. Columns: 21 viewpoints (7 yaws x 3 pitches). Green boxes mark MediaPipe detections; white panels mean no detection.

In [5]:
TAGS = ['original', 'delete', 'noise']
ROW_TITLES = ['Original', 'Vertex deletion', 'Noise perturbation']

view_keys = [(y, p) for y in YAWS for p in PITCHES]
n_cols = len(view_keys)

fig, axes = plt.subplots(
    len(TAGS), n_cols,
    figsize=(1.6 * n_cols, 1.6 * len(TAGS)),
)
for r, tag in enumerate(TAGS):
    for c, (yaw, pitch) in enumerate(view_keys):
        ax = axes[r, c]
        path = annotated_paths[tag][(yaw, pitch)]
        img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0:
            ax.set_title(f'y{yaw:+d}/p{pitch:+d}', fontsize=6)
        if c == 0:
            ax.set_ylabel(ROW_TITLES[r], fontsize=10)
fig.suptitle(f'{s_id(SUBJECT)} (Subject{SUBJECT}) - MediaPipe detections on every view')
fig.tight_layout()
out = OUT_DIR / 's8_mediapipe_grid.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
plt.close(fig)
print(f'Wrote {out}')

Wrote thesis_results_out/s8_mediapipe_boxes/s8_mediapipe_grid.png


## 5. Per-variant summary

In [6]:
n_views = len(YAWS) * len(PITCHES)
for tag in TAGS:
    sub = df[df.variant == tag] if len(df) else df
    n_hit_views = sub.groupby(['yaw', 'pitch']).size().shape[0] if len(sub) else 0
    n_dets = len(sub)
    max_score = float(sub.score.max()) if len(sub) else 0.0
    print(f'{tag:10s}  views_with_hit={n_hit_views:>2d}/{n_views}  '
          f'total_detections={n_dets:>2d}  max_score={max_score:.3f}')

original    views_with_hit=18/21  total_detections=20  max_score=0.946
delete      views_with_hit=15/21  total_detections=15  max_score=0.819
noise       views_with_hit=13/21  total_detections=13  max_score=0.797


In [ ]:
## Figure for the thesis (Subject 17 / S2 only, per data-sharing rule).
##
## The cell above produced the full keypoint diagnostic on Subject 12 (S8) for
## by-eye verification - that diagnostic stays as supplementary analysis. The
## thesis figure is restricted to S2 (Subject 17) and is composed below from
## annotated S2 renders saved in
## `thesis_results_out/s2_keypoint_diagnostics/`.
##
## Both the bare-cap S8 case and the optode-cap S2 case show the same pattern:
## post-deletion residual hits land on bald-skull silhouettes with clustered,
## anatomically inconsistent keypoint placements.

import shutil

S2_DIAG = pathlib.Path('thesis_results_out/s2_keypoint_diagnostics')
THESIS_FIG_DIR = pathlib.Path(
    '/home/ma7/BA/Thesis_template-master-LaTeX_template_thesis/'
    'LaTeX_template_thesis/Figures/results'
)

panels = [
    (S2_DIAG / 'A_real_face_S2.png',
     r'Original, yaw $0^{\circ}$, pitch $-20^{\circ}$' '\n' r'score 0.82 - real face'),
    (S2_DIAG / 'B_hallucination_S2.png',
     r'Deletion, yaw $+60^{\circ}$, pitch $-20^{\circ}$' '\n' r'score 0.52 - hallucination'),
]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, (path, title) in zip(axes, panels):
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=10)
fig.tight_layout()

out = THESIS_FIG_DIR / 'detectability_keypoints_S2.png'
fig.savefig(out, dpi=200, bbox_inches='tight')
plt.close(fig)
print(f'Wrote {out}')

## 6. Two-panel keypoint diagnostic for the thesis

A 2-panel figure for §4.4.4: a real-face hit (frontal, original) where the BlazeFace 6 keypoints land on real eyes / nose / mouth / ears, side-by-side with a hallucination (left profile, deletion variant) where the keypoints cluster onto the bald-skull silhouette in an anatomically impossible configuration (nose tip above eyes, mouth at eye level). Composed from the previously-saved `keypoint_diagnostics/A_*` and `keypoint_diagnostics/C_*` PNGs. Saves directly into the LaTeX `Figures/results/` tree (`feedback_pngs_in_latex_dir`).